<a href="https://colab.research.google.com/github/junseok-jay/guitar_CV/blob/main/test_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install -q ultralytics

Fri Apr  3 12:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path

yaml_path = Path(f"{dataset_dir}/data.yaml")
text = yaml_path.read_text(encoding="utf-8")
text = text.replace("train: ../train/images", "train: train/images")
text = text.replace("val: ../valid/images", "val: valid/images")
text = text.replace("test: ../test/images", "test: test/images")
yaml_path.write_text(text, encoding="utf-8")
print(yaml_path.read_text(encoding="utf-8"))

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['neck']

roboflow:
  workspace: s-workspace-y3mjn
  project: guitar-neck-detection-suhgk-uemtb
  version: 3
  license: CC BY 4.0
  url: https://universe.roboflow.com/s-workspace-y3mjn/guitar-neck-detection-suhgk-uemtb/dataset/3


In [14]:
from ultralytics import YOLO
from pathlib import Path

yaml_path = "handtip/data.yaml"

# Fix: Remove roboflow metadata from data.yaml as it causes a YAML syntax error
yaml_file_path = Path(yaml_path)
text = yaml_file_path.read_text(encoding="utf-8")

roboflow_start_index = text.find("roboflow:")
if roboflow_start_index != -1:
    text = text[:roboflow_start_index].strip()

yaml_file_path.write_text(text, encoding="utf-8")
print("Modified data.yaml content:")
print(yaml_file_path.read_text(encoding="utf-8"))

model = YOLO("yolo11s-pose.pt")
# model.train(
#     data=yaml_path,
#     epochs=100,
#     imgsz=640,
#     batch=8,
#     project="hand_tip",
#     name="train"
# )

model.train(
    data=yaml_path,
    epochs=120,
    imgsz=640,
    batch=16,
    project="hand_tip",
    name="train",
    degrees=10,
    scale=0.2,
    translate=0.1,
    fliplr=0.5,
    mosaic=0.0,
    close_mosaic=0,
)

Modified data.yaml content:
train: ../train/images
val: ../valid/images
test: ../test/images

kpt_shape: [21, 3]
flip_idx: [0, 17, 18, 19, 20, 13, 14, 15, 16, 9, 10, 11, 12, 5, 6, 7, 8, 1, 2, 3, 4]

nc: 1
names: ['hand']
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=handtip/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=30

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79512a0d2f00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034, 

In [15]:
from ultralytics import YOLO

best_model = YOLO("runs/pose/hand_tip/train/weights/best.pt")
metrics = best_model.val(data="handtip/data.yaml")
print(metrics)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-pose summary (fused): 110 layers, 2,956,000 parameters, 0 gradients, 7.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 281.5±85.6 MB/s, size: 6.5 KB)
val: Scanning /content/handtip/valid/labels.cache... 326 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 326/326 76.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.9it/s 7.2s
                   all        326        326      0.994      0.997      0.995      0.866      0.793      0.785      0.744      0.393
Speed: 4.0ms preprocess, 4.9ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/runs/pose/val4
ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.Confusion

In [16]:
!pwd
!find runs/pose/hand_tip/train/weights/best.pt -maxdepth 2 -type f | sort

/content
runs/pose/hand_tip/train/weights/best.pt


In [17]:
from google.colab import files
files.download("runs/pose/hand_tip/train/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>